In [ ]:
# ============================================================

# DRWEIBO EARLY-WINDOW EXPERIMENT SUITE

#

# Sparse Conversation Structure Modeling

# for Early Rumor Verification

#

# Controlled development variant:

#   RoBERTa/BERT + GAT + SSEE + SAF

#

# V2 change:

#   Add feature-space alignment before adaptive fusion:

#

#   semantic_rep  -> Linear + LayerNorm -> h_x_aligned

#   structural_rep-> Linear + LayerNorm -> h_e_aligned

#

#   h_f = lambda * h_e_aligned + (1-lambda) * h_x_aligned

#

# IMPORTANT:

#   GAT / SEM / AGCC / ASE / data split / seed / optimizer

#   protocol are unchanged from the previous full-model run.

# ============================================================

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import random
from pathlib import Path
from collections import Counter, deque

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from transformers import (
    BertTokenizer,
    BertModel,
    RobertaTokenizer,
    RobertaModel,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from tqdm import tqdm


# ============================================================

# 1. PATH CONFIG

# ============================================================

BASE_DIR = Path("/root/autodl-fs/processed_scsr")

SPLIT_DIR = BASE_DIR / "splits"

OUTPUT_DIR = (
    BASE_DIR /
    "experiment_results" /
    "full_model_v3"
)


# ============================================================

# 2. DATASET

# ============================================================

DATASET = "DRWeibo"

# Later:

# DATASET = "PHEME"

PHEME_FOLD = 1


# ============================================================

# 3. LOCAL PRETRAINED MODELS

# ============================================================

DRWEIBO_MODEL_DIR = (
    BASE_DIR /
    "chinese_roberta_wwm_ext"
)

PHEME_MODEL_DIR = (
    BASE_DIR /
    "roberta_base"
)


# ============================================================

# 4. REPRODUCIBILITY

# ============================================================

SEED = 42


# ============================================================

# 5. TEXT ENCODER

# ============================================================

MAX_LENGTH = 128
NODE_CHUNK_SIZE = 8


# ============================================================

# 6. BATCH

# ============================================================

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8


# ============================================================

# 7. GAT

# ============================================================

GAT_HIDDEN_DIM = 256
GAT_HEADS = 4

assert GAT_HIDDEN_DIM % GAT_HEADS == 0

GAT_DROPOUT = 0.3
GAT_ATTENTION_DROPOUT = 0.2
LEAKY_RELU_SLOPE = 0.2

MAKE_BIDIRECTIONAL = True


# ============================================================

# 8. SSEE

# ============================================================

SEM_EVIDENCE_DIM = 128
GRAPH_STAT_DIM = 6
SSEE_DROPOUT = 0.3


# ============================================================

# 9. SAF

# ============================================================

SPARSITY_HIDDEN_DIM = 64
SAF_DROPOUT = 0.2

# Hidden dimension of the representation-aware fusion gate

FUSION_GATE_HIDDEN_DIM = 128

# Common fusion-space dimensionality.

FUSION_DIM = GAT_HIDDEN_DIM


# ============================================================

# 10. OPTIMIZATION

# ============================================================

ENCODER_LR = 2e-5
NEW_MODULE_LR = 1e-3

WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
GRADIENT_CLIP = 1.0


# ============================================================

# 11. TRAINING

# ============================================================

MAX_EPOCHS = 10
PATIENCE = 3
NUM_WORKERS = 0


# ============================================================

# 12. RANDOM SEED

# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================

# 13. DEVICE

# ============================================================

def get_device():

    if torch.cuda.is_available():
    
        device = torch.device("cuda")
    
        prop = torch.cuda.get_device_properties(0)
    
        total_memory = (
            prop.total_memory /
            1024 ** 3
        )
    
        print("\nCUDA available")
        print("GPU:", torch.cuda.get_device_name(0))
        print(f"Total VRAM: {total_memory:.2f} GB")
    
        torch.backends.cuda.matmul.allow_tf32 = True
    
        return device
    
    print("\nCUDA unavailable. Using CPU.")
    
    return torch.device("cpu")


# ============================================================

# 14. CUDA MEMORY

# ============================================================

def print_cuda_memory():

    if not torch.cuda.is_available():
        return
    
    allocated = (
        torch.cuda.memory_allocated() /
        1024 ** 3
    )
    
    reserved = (
        torch.cuda.memory_reserved() /
        1024 ** 3
    )
    
    peak = (
        torch.cuda.max_memory_allocated() /
        1024 ** 3
    )
    
    print(
        f"CUDA memory | "
        f"allocated={allocated:.2f} GB | "
        f"reserved={reserved:.2f} GB | "
        f"peak={peak:.2f} GB"
    )


# ============================================================

# 15. LOAD JSONL

# ============================================================

def load_jsonl(path):

    samples = []
    
    with open(path, "r", encoding="utf-8") as f:
    
        for line in f:
    
            line = line.strip()
    
            if not line:
                continue
    
            samples.append(
                json.loads(line)
            )
    
    return samples


# ============================================================

# 16. LABEL MAPPING

# ============================================================

def get_label_mapping(dataset):

    if dataset == "DRWeibo":
    
        return (
            {
                "0": 0,
                "1": 1,
            },
            {
                0: "0",
                1: "1",
            },
        )
    
    if dataset == "PHEME":
    
        return (
            {
                "false": 0,
                "true": 1,
                "unverified": 2,
            },
            {
                0: "false",
                1: "true",
                2: "unverified",
            },
        )
    
    raise ValueError(
        f"Unsupported dataset: {dataset}"
    )


# ============================================================

# 17. PATHS

# ============================================================

def get_paths():

    if DATASET == "DRWeibo":
    
        data_dir = SPLIT_DIR / "DRWeibo"
        model_dir = DRWEIBO_MODEL_DIR
    
    elif DATASET == "PHEME":
    
        data_dir = (
            SPLIT_DIR /
            "PHEME" /
            f"fold_{PHEME_FOLD}"
        )
    
        model_dir = PHEME_MODEL_DIR
    
    else:
    
        raise ValueError(DATASET)
    
    return (
        data_dir / "train.jsonl",
        data_dir / "val.jsonl",
        data_dir / "test.jsonl",
        model_dir,
    )


# ============================================================

# 18. TOKENIZER

# ============================================================

def load_tokenizer(dataset, model_dir):

    if dataset == "DRWeibo":
    
        return BertTokenizer.from_pretrained(
            str(model_dir),
            local_files_only=True
        )
    
    if dataset == "PHEME":
    
        return RobertaTokenizer.from_pretrained(
            str(model_dir),
            local_files_only=True
        )
    
    raise ValueError(dataset)


# ============================================================

# 19. GRAPH DEPTH

# ============================================================

def compute_max_depth(
    nodes,
    edges,
    root_id
):

    node_ids = {
        str(node["node_id"])
        for node in nodes
    }
    
    children = {
        node_id: []
        for node_id in node_ids
    }
    
    for edge in edges:
    
        if (
            not isinstance(edge, (list, tuple))
            or
            len(edge) != 2
        ):
            continue
    
        parent = str(edge[0])
        child = str(edge[1])
    
        if (
            parent in node_ids
            and
            child in node_ids
        ):
            children[parent].append(child)
    
    root_id = str(root_id)
    
    if root_id not in node_ids:
        return 0
    
    queue = deque(
        [(root_id, 0)]
    )
    
    visited = set()
    max_depth = 0
    
    while queue:
    
        node_id, depth = queue.popleft()
    
        if node_id in visited:
            continue
    
        visited.add(node_id)
    
        max_depth = max(
            max_depth,
            depth
        )
    
        for child in children.get(
            node_id,
            []
        ):
            queue.append(
                (child, depth + 1)
            )
    
    return max_depth


# ============================================================

# 20. GRAPH STATISTICS / SSD INPUT

# ============================================================

def extract_graph_statistics(sample):

    nodes = sample.get(
        "nodes",
        []
    )
    
    edges = sample.get(
        "edges",
        []
    )
    
    n = len(nodes)
    e = len(edges)
    
    existing = sample.get(
        "graph_statistics",
        {}
    )
    
    depth = existing.get(
        "max_depth",
        existing.get(
            "depth",
            None
        )
    )
    
    if depth is None:
    
        root_id = sample.get(
            "root_id",
            (
                nodes[0]["node_id"]
                if nodes
                else ""
            )
        )
    
        depth = compute_max_depth(
            nodes,
            edges,
            root_id
        )
    
    # Directed graph density.
    if n > 1:
        density = (
            e /
            (
                n *
                (n - 1)
            )
        )
    else:
        density = 0.0
    
    # Average degree.
    if n > 0:
        avg_degree = (
            2.0 *
            e /
            n
        )
    else:
        avg_degree = 0.0
    
    # Same working definition used in the previous SSEE/full run.
    branching = (
        e /
        max(
            float(depth),
            1.0
        )
    )
    
    # Fixed numerical scaling.
    stats = np.array(
        [
            np.log1p(n),
            np.log1p(e),
            np.log1p(float(depth)),
            float(density),
            np.log1p(avg_degree),
            np.log1p(branching),
        ],
        dtype=np.float32
    )
    
    return stats


# ============================================================

# 21. DATASET

# ============================================================

class ConversationGraphDataset(
    Dataset
):

    def __init__(
        self,
        samples,
        label2id
    ):
    
        self.samples = samples
        self.label2id = label2id


    def __len__(self):
    
        return len(
            self.samples
        )


    def __getitem__(
        self,
        idx
    ):
    
        sample = self.samples[idx]
    
        nodes = sample.get(
            "nodes",
            []
        )
    
        edges = sample.get(
            "edges",
            []
        )
    
        node_id_to_idx = {}
        texts = []
    
        for node_idx, node in enumerate(
            nodes
        ):
    
            node_id = str(
                node["node_id"]
            )
    
            node_id_to_idx[
                node_id
            ] = node_idx
    
            text = str(
                node.get(
                    "text",
                    ""
                )
            ).strip()
    
            if not text:
                text = "[EMPTY]"
    
            texts.append(text)
    
        indexed_edges = []
    
        for edge in edges:
    
            if (
                not isinstance(edge, (list, tuple))
                or
                len(edge) != 2
            ):
                continue
    
            parent = str(edge[0])
            child = str(edge[1])
    
            if (
                parent in node_id_to_idx
                and
                child in node_id_to_idx
            ):
    
                indexed_edges.append(
                    (
                        node_id_to_idx[parent],
                        node_id_to_idx[child],
                    )
                )
    
        label = self.label2id[
            str(
                sample["label"]
            )
        ]
    
        graph_stats = (
            extract_graph_statistics(
                sample
            )
        )
    
        return {
            "id": str(sample["id"]),
            "texts": texts,
            "edges": indexed_edges,
            "label": label,
            "graph_stats": graph_stats,
        }


# ============================================================

# 22. COLLATOR

# ============================================================

class GraphConversationCollator:

    def __init__(
        self,
        tokenizer,
        max_length
    ):
    
        self.tokenizer = tokenizer
        self.max_length = max_length


    def __call__(
        self,
        batch
    ):
    
        all_texts = []
        all_edges = []
        conversation_ids = []
        labels = []
        sample_ids = []
        graph_stats = []
    
        node_offset = 0
    
        for conv_idx, item in enumerate(
            batch
        ):
    
            texts = item["texts"]
            edges = item["edges"]
    
            num_nodes = len(texts)
    
            if num_nodes == 0:
    
                raise ValueError(
                    f"Conversation "
                    f"{item['id']} "
                    f"contains zero nodes."
                )
    
            all_texts.extend(texts)
    
            conversation_ids.extend(
                [conv_idx] *
                num_nodes
            )
    
            for src, dst in edges:
    
                all_edges.append(
                    (
                        src + node_offset,
                        dst + node_offset,
                    )
                )
    
            node_offset += num_nodes
    
            labels.append(
                item["label"]
            )
    
            sample_ids.append(
                item["id"]
            )
    
            graph_stats.append(
                item["graph_stats"]
            )
    
        encoded = self.tokenizer(
            all_texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
    
        if all_edges:
    
            edge_index = (
                torch.tensor(
                    all_edges,
                    dtype=torch.long
                )
                .t()
                .contiguous()
            )
    
        else:
    
            edge_index = torch.empty(
                (2, 0),
                dtype=torch.long
            )
    
        return {
            "input_ids":
                encoded["input_ids"],
    
            "attention_mask":
                encoded["attention_mask"],
    
            "edge_index":
                edge_index,
    
            "conversation_ids":
                torch.tensor(
                    conversation_ids,
                    dtype=torch.long
                ),
    
            "graph_stats":
                torch.tensor(
                    np.stack(
                        graph_stats
                    ),
                    dtype=torch.float32
                ),
    
            "labels":
                torch.tensor(
                    labels,
                    dtype=torch.long
                ),
    
            "sample_ids":
                sample_ids,
        }


# ============================================================

# 23. GRAPH EDGE PREPARATION

# ============================================================

def prepare_edge_index(
    edge_index,
    num_nodes,
    make_bidirectional=True
):

    edges = edge_index
    device = edge_index.device
    
    if (
        make_bidirectional
        and
        edges.size(1) > 0
    ):
    
        reverse = torch.stack(
            [
                edges[1],
                edges[0],
            ],
            dim=0
        )
    
        edges = torch.cat(
            [
                edges,
                reverse,
            ],
            dim=1
        )
    
    node_idx = torch.arange(
        num_nodes,
        dtype=torch.long,
        device=device
    )
    
    self_loops = torch.stack(
        [
            node_idx,
            node_idx,
        ],
        dim=0
    )
    
    edges = torch.cat(
        [
            edges,
            self_loops,
        ],
        dim=1
    )
    
    edge_pairs = (
        edges
        .t()
        .contiguous()
    )
    
    edge_pairs = torch.unique(
        edge_pairs,
        dim=0
    )
    
    return (
        edge_pairs
        .t()
        .contiguous()
    )


# ============================================================

# 24. MULTI-HEAD EDGE SOFTMAX

# ============================================================

def edge_softmax(
    scores,
    dst,
    num_nodes
):

    scores_fp32 = scores.float()
    
    num_heads = (
        scores_fp32.size(1)
    )
    
    expanded_dst = (
        dst
        .unsqueeze(1)
        .expand(
            -1,
            num_heads
        )
    )
    
    max_per_node = torch.full(
        (
            num_nodes,
            num_heads
        ),
        -float("inf"),
        dtype=torch.float32,
        device=scores.device
    )
    
    max_per_node.scatter_reduce_(
        0,
        expanded_dst,
        scores_fp32,
        reduce="amax",
        include_self=True
    )
    
    stabilized = (
        scores_fp32
        -
        max_per_node[dst]
    )
    
    exp_scores = torch.exp(
        stabilized
    )
    
    denominator = torch.zeros(
        (
            num_nodes,
            num_heads
        ),
        dtype=torch.float32,
        device=scores.device
    )
    
    denominator.index_add_(
        0,
        dst,
        exp_scores
    )
    
    return (
        exp_scores /
        (
            denominator[dst]
            +
            1e-12
        )
    )


# ============================================================

# 25. SCALAR EDGE SOFTMAX

# ============================================================

def scalar_edge_softmax(
    scores,
    dst,
    num_nodes
):

    scores_fp32 = scores.float()
    
    max_per_node = torch.full(
        (num_nodes,),
        -float("inf"),
        dtype=torch.float32,
        device=scores.device
    )
    
    max_per_node.scatter_reduce_(
        0,
        dst,
        scores_fp32,
        reduce="amax",
        include_self=True
    )
    
    stabilized = (
        scores_fp32
        -
        max_per_node[dst]
    )
    
    exp_scores = torch.exp(
        stabilized
    )
    
    denominator = torch.zeros(
        (num_nodes,),
        dtype=torch.float32,
        device=scores.device
    )
    
    denominator.index_add_(
        0,
        dst,
        exp_scores
    )
    
    return (
        exp_scores /
        (
            denominator[dst]
            +
            1e-12
        )
    )


# ============================================================

# 26. VANILLA GAT LAYER

# ============================================================

class VanillaGATLayer(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        output_dim,
        num_heads,
        dropout,
        attention_dropout,
        negative_slope=0.2
    ):
    
        super().__init__()
    
        assert (
            output_dim %
            num_heads ==
            0
        )
    
        self.output_dim = output_dim
        self.num_heads = num_heads
    
        self.head_dim = (
            output_dim //
            num_heads
        )
    
        self.negative_slope = negative_slope
    
        self.linear = nn.Linear(
            input_dim,
            output_dim,
            bias=False
        )
    
        self.att_src = nn.Parameter(
            torch.empty(
                num_heads,
                self.head_dim
            )
        )
    
        self.att_dst = nn.Parameter(
            torch.empty(
                num_heads,
                self.head_dim
            )
        )
    
        self.bias = nn.Parameter(
            torch.zeros(
                output_dim
            )
        )
    
        self.feature_dropout = nn.Dropout(
            dropout
        )
    
        self.attention_dropout = nn.Dropout(
            attention_dropout
        )
    
        self.reset_parameters()


    def reset_parameters(self):
    
        nn.init.xavier_uniform_(
            self.linear.weight
        )
    
        nn.init.xavier_uniform_(
            self.att_src
        )
    
        nn.init.xavier_uniform_(
            self.att_dst
        )
    
        nn.init.zeros_(
            self.bias
        )


    def forward(
        self,
        x,
        edge_index
    ):
    
        num_nodes = x.size(0)
    
        x = self.feature_dropout(x)
    
        h = self.linear(x)
    
        h = h.view(
            num_nodes,
            self.num_heads,
            self.head_dim
        )
    
        src = edge_index[0]
        dst = edge_index[1]
    
        src_score = (
            (
                h[src]
                *
                self.att_src
            )
            .sum(dim=-1)
        )
    
        dst_score = (
            (
                h[dst]
                *
                self.att_dst
            )
            .sum(dim=-1)
        )
    
        scores = F.leaky_relu(
            src_score + dst_score,
            negative_slope=
                self.negative_slope
        )
    
        alpha = edge_softmax(
            scores,
            dst,
            num_nodes
        )
    
        alpha = self.attention_dropout(
            alpha
        )
    
        # AMP dtype alignment
        alpha = alpha.to(
            dtype=h.dtype
        )
    
        messages = (
            h[src]
            *
            alpha.unsqueeze(-1)
        )
    
        output = torch.zeros(
            (
                num_nodes,
                self.num_heads,
                self.head_dim
            ),
            dtype=h.dtype,
            device=h.device
        )
    
        output.index_add_(
            0,
            dst,
            messages
        )
    
        output = output.reshape(
            num_nodes,
            self.output_dim
        )
    
        return (
            output +
            self.bias
        )


# ============================================================

# 27. SEM

# ============================================================

class StructuralEvidenceMining(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        evidence_dim,
        dropout
    ):
    
        super().__init__()
    
        self.query_proj = nn.Linear(
            hidden_dim,
            evidence_dim,
            bias=False
        )
    
        self.key_proj = nn.Linear(
            hidden_dim,
            evidence_dim,
            bias=False
        )
    
        self.evidence_vector = nn.Parameter(
            torch.empty(
                evidence_dim
            )
        )
    
        self.dropout = nn.Dropout(
            dropout
        )
    
        self.reset_parameters()


    def reset_parameters(self):
    
        nn.init.xavier_uniform_(
            self.query_proj.weight
        )
    
        nn.init.xavier_uniform_(
            self.key_proj.weight
        )
    
        nn.init.normal_(
            self.evidence_vector,
            mean=0.0,
            std=0.02
        )


    def forward(
        self,
        h_g,
        edge_index
    ):
    
        num_nodes = h_g.size(0)
    
        src = edge_index[0]
        dst = edge_index[1]
    
        q_i = self.query_proj(
            h_g[dst]
        )
    
        k_j = self.key_proj(
            h_g[src]
        )
    
        evidence_hidden = torch.tanh(
            q_i + k_j
        )
    
        evidence_scores = (
            evidence_hidden
            *
            self.evidence_vector
        ).sum(dim=-1)
    
        alpha = scalar_edge_softmax(
            evidence_scores,
            dst,
            num_nodes
        )
    
        alpha = self.dropout(
            alpha
        )
    
        alpha = alpha.to(
            dtype=h_g.dtype
        )
    
        messages = (
            h_g[src]
            *
            alpha.unsqueeze(-1)
        )
    
        h_m = torch.zeros(
            h_g.shape,
            dtype=h_g.dtype,
            device=h_g.device
        )
    
        h_m.index_add_(
            0,
            dst,
            messages
        )
    
        return h_m


# ============================================================

# 28. AGCC

# ============================================================

class AdaptiveGlobalContextCompensation(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        stat_dim
    ):
    
        super().__init__()
    
        self.gate = nn.Linear(
            hidden_dim * 2 +
            stat_dim,
            hidden_dim
        )


    def forward(
        self,
        h_m,
        conversation_ids,
        graph_stats,
        batch_size
    ):
    
        hidden_dim = h_m.size(-1)
    
        global_context = torch.zeros(
            (
                batch_size,
                hidden_dim
            ),
            dtype=h_m.dtype,
            device=h_m.device
        )
    
        global_context.index_add_(
            0,
            conversation_ids,
            h_m
        )
    
        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=h_m.dtype,
                device=h_m.device
            )
        )
    
        global_context = (
            global_context /
            counts
        )
    
        node_global = (
            global_context[
                conversation_ids
            ]
        )
    
        node_stats = (
            graph_stats[
                conversation_ids
            ]
            .to(
                dtype=h_m.dtype
            )
        )
    
        gate_input = torch.cat(
            [
                h_m,
                node_global,
                node_stats,
            ],
            dim=-1
        )
    
        gamma = torch.sigmoid(
            self.gate(
                gate_input
            )
        )
    
        h_c = (
            h_m +
            gamma *
            node_global
        )
    
        return h_c


# ============================================================

# 29. ASE

# ============================================================

class AdaptiveStructuralEnhancement(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        stat_dim,
        dropout
    ):
    
        super().__init__()
    
        self.mlp = nn.Sequential(
            nn.Linear(
                hidden_dim +
                stat_dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
        )


    def forward(
        self,
        h_c,
        h_g,
        conversation_ids,
        graph_stats
    ):
    
        node_stats = (
            graph_stats[
                conversation_ids
            ]
            .to(
                dtype=h_c.dtype
            )
        )
    
        ase_input = torch.cat(
            [
                h_c,
                node_stats,
            ],
            dim=-1
        )
    
        correction = self.mlp(
            ase_input
        )
    
        # Residual to initial GAT structural representation.
        h_e = (
            h_g +
            correction
        )
    
        return h_e



# ============================================================

# 30. CONTROLLED MODEL VARIANTS

# ============================================================

class SemanticOnlyClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()
    
        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)
    
        self.encoder.gradient_checkpointing_enable()
    
        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False
    
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(
            self.encoder.config.hidden_size,
            num_classes
        )
    
    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)
    
        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)
    
            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )
    
            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )
    
        return torch.cat(chunks, dim=0)
    
    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)
    
        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )
    
        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )
    
        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )
    
        return pooled / counts
    
    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )
    
        h_graph = self.conversation_mean_pool(
            h_x,
            conversation_ids,
            batch_size
        )
    
        return self.classifier(
            self.dropout(h_graph)
        )


class GATOnlyClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()
    
        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)
    
        self.encoder.gradient_checkpointing_enable()
    
        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False
    
        encoder_dim = self.encoder.config.hidden_size
    
        self.gat1 = VanillaGATLayer(
            input_dim=encoder_dim,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )
    
        self.gat2 = VanillaGATLayer(
            input_dim=GAT_HIDDEN_DIM,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )
    
        self.dropout = nn.Dropout(DROPOUT)
    
        self.classifier = nn.Linear(
            GAT_HIDDEN_DIM,
            num_classes
        )
    
    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)
    
        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)
    
            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )
    
            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )
    
        return torch.cat(chunks, dim=0)
    
    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)
    
        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )
    
        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )
    
        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )
    
        return pooled / counts
    
    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )
    
        num_nodes = h_x.size(0)
    
        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )
    
        h_g = F.elu(
            self.gat1(
                h_x,
                graph_edge_index
            )
        )
    
        h_g = F.elu(
            self.gat2(
                h_g,
                graph_edge_index
            )
        )
    
        h_graph = self.conversation_mean_pool(
            h_g,
            conversation_ids,
            batch_size
        )
    
        return self.classifier(
            self.dropout(h_graph)
        )


class GATSSEEClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()
    
        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)
    
        self.encoder.gradient_checkpointing_enable()
    
        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False
    
        encoder_dim = self.encoder.config.hidden_size
    
        self.gat1 = VanillaGATLayer(
            input_dim=encoder_dim,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )
    
        self.gat2 = VanillaGATLayer(
            input_dim=GAT_HIDDEN_DIM,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )
    
        self.sem = StructuralEvidenceMining(
            hidden_dim=GAT_HIDDEN_DIM,
            evidence_dim=SEM_EVIDENCE_DIM,
            dropout=SSEE_DROPOUT
        )
    
        self.agcc = AdaptiveGlobalContextCompensation(
            hidden_dim=GAT_HIDDEN_DIM,
            stat_dim=GRAPH_STAT_DIM
        )
    
        self.ase = AdaptiveStructuralEnhancement(
            hidden_dim=GAT_HIDDEN_DIM,
            stat_dim=GRAPH_STAT_DIM,
            dropout=SSEE_DROPOUT
        )
    
        self.dropout = nn.Dropout(DROPOUT)
    
        self.classifier = nn.Linear(
            GAT_HIDDEN_DIM,
            num_classes
        )
    
    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)
    
        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)
    
            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )
    
            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )
    
        return torch.cat(chunks, dim=0)
    
    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)
    
        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )
    
        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )
    
        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )
    
        return pooled / counts
    
    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )
    
        num_nodes = h_x.size(0)
    
        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )
    
        h_g = F.elu(
            self.gat1(
                h_x,
                graph_edge_index
            )
        )
    
        h_g = F.elu(
            self.gat2(
                h_g,
                graph_edge_index
            )
        )
    
        h_m = self.sem(
            h_g,
            graph_edge_index
        )
    
        h_c = self.agcc(
            h_m,
            conversation_ids,
            graph_stats,
            batch_size
        )
    
        h_e = self.ase(
            h_c,
            h_g,
            conversation_ids,
            graph_stats
        )
    
        h_graph = self.conversation_mean_pool(
            h_e,
            conversation_ids,
            batch_size
        )
    
        return self.classifier(
            self.dropout(h_graph)
        )


# ============================================================

# 31. EARLY-WINDOW EXPERIMENT CONFIG

# ============================================================

WINDOWS = [10, 30, 60, 120, 240]

MODEL_VARIANTS = [
    "semantic",
    "gat",
    "ssee",
]

# To test SSEE only first:

# MODEL_VARIANTS = ["ssee"]

EARLY_DATA_DIR = (
    BASE_DIR /
    "DRWeibo"
)

EARLY_RESULTS_DIR = (
    BASE_DIR /
    "experiment_results" /
    "drweibo_early_windows"
)


# ============================================================

# 32. EARLY-WINDOW FILE

# ============================================================

def get_early_window_path(window):

    candidates = [
        EARLY_DATA_DIR /
        f"drweibo_{window}min.jsonl",
    
        BASE_DIR /
        f"drweibo_{window}min.jsonl",
    
        BASE_DIR /
        "DRWeibo" /
        f"DRWeibo_{window}min.jsonl",
    
        BASE_DIR /
        "DRWeibo" /
        f"drweibo_{window}_min.jsonl",
    ]
    
    for path in candidates:
    
        if path.exists():
            return path
    
    searched = "\n".join(
        str(path)
        for path in candidates
    )
    
    raise FileNotFoundError(
        f"Cannot find {window}-min DRWeibo file.\n"
        f"Searched:\n{searched}"
    )


# ============================================================

# 33. LOCKED SPLIT IDS

# ============================================================

def load_fixed_split_ids():

    split_dir = (
        SPLIT_DIR /
        "DRWeibo"
    )
    
    split_ids = {}
    
    for split_name in [
        "train",
        "val",
        "test",
    ]:
    
        path = (
            split_dir /
            f"{split_name}.jsonl"
        )
    
        samples = load_jsonl(
            path
        )
    
        ids = [
            str(sample["id"])
            for sample in samples
        ]
    
        if len(ids) != len(set(ids)):
    
            raise ValueError(
                f"Duplicate IDs found in {path}"
            )
    
        split_ids[
            split_name
        ] = ids
    
    all_ids = (
        split_ids["train"]
        +
        split_ids["val"]
        +
        split_ids["test"]
    )
    
    if len(all_ids) != len(set(all_ids)):
    
        raise ValueError(
            "Train/val/test ID overlap detected."
        )
    
    return split_ids


# ============================================================

# 34. APPLY LOCKED SPLIT TO EARLY WINDOW

# ============================================================

def build_window_splits(
    window_path,
    split_ids
):

    samples = load_jsonl(
        window_path
    )
    
    sample_map = {}
    
    for sample in samples:
    
        sample_id = str(
            sample["id"]
        )
    
        if sample_id in sample_map:
    
            raise ValueError(
                f"Duplicate ID in "
                f"{window_path}: "
                f"{sample_id}"
            )
    
        sample_map[
            sample_id
        ] = sample
    
    window_splits = {}
    
    for split_name in [
        "train",
        "val",
        "test",
    ]:
    
        wanted_ids = (
            split_ids[
                split_name
            ]
        )
    
        missing = [
            sample_id
            for sample_id in wanted_ids
            if sample_id not in sample_map
        ]
    
        if missing:
    
            raise ValueError(
                f"{window_path.name}: "
                f"{len(missing)} IDs missing from "
                f"{split_name}. Examples: "
                f"{missing[:10]}"
            )
    
        window_splits[
            split_name
        ] = [
            sample_map[
                sample_id
            ]
            for sample_id in wanted_ids
        ]
    
    return window_splits


# ============================================================

# 35. METRICS

# ============================================================

def calculate_metrics(
    labels,
    predictions,
    id2label
):

    label_ids = list(
        id2label.keys()
    )
    
    accuracy = accuracy_score(
        labels,
        predictions
    )
    
    macro_f1 = f1_score(
        labels,
        predictions,
        labels=label_ids,
        average="macro",
        zero_division=0
    )
    
    class_f1 = f1_score(
        labels,
        predictions,
        labels=label_ids,
        average=None,
        zero_division=0
    )
    
    report = classification_report(
        labels,
        predictions,
        labels=label_ids,
        target_names=[
            id2label[i]
            for i in label_ids
        ],
        output_dict=True,
        zero_division=0
    )
    
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "class_f1": class_f1,
        "report": report,
    }


# ============================================================

# 36. EVALUATION

# ============================================================

@torch.no_grad()
def evaluate(
    model,
    dataloader,
    device,
    criterion,
    id2label,
    use_amp
):

    model.eval()
    
    total_loss = 0.0
    
    all_labels = []
    all_predictions = []
    all_sample_ids = []
    
    for batch in tqdm(
        dataloader,
        desc="Evaluating",
        leave=False
    ):
    
        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )
    
        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )
    
        edge_index = batch["edge_index"].to(
            device,
            non_blocking=True
        )
    
        conversation_ids = batch["conversation_ids"].to(
            device,
            non_blocking=True
        )
    
        graph_stats = batch["graph_stats"].to(
            device,
            non_blocking=True
        )
    
        labels = batch["labels"].to(
            device,
            non_blocking=True
        )
    
        batch_size = labels.size(0)
    
        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):
    
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                edge_index=edge_index,
                conversation_ids=conversation_ids,
                graph_stats=graph_stats,
                batch_size=batch_size
            )
    
            loss = criterion(
                logits,
                labels
            )
    
        total_loss += loss.item()
    
        predictions = torch.argmax(
            logits,
            dim=1
        )
    
        all_labels.extend(
            labels.cpu().tolist()
        )
    
        all_predictions.extend(
            predictions.cpu().tolist()
        )
    
        all_sample_ids.extend(
            batch["sample_ids"]
        )
    
    metrics = calculate_metrics(
        all_labels,
        all_predictions,
        id2label
    )
    
    return {
        "loss":
            total_loss /
            max(len(dataloader), 1),
    
        **metrics,
    
        "labels":
            all_labels,
    
        "predictions":
            all_predictions,
    
        "sample_ids":
            all_sample_ids,
    }


# ============================================================

# 37. OPTIMIZER STEP

# ============================================================

def optimizer_update(
    model,
    optimizer,
    scaler,
    use_amp
):

    if use_amp:
    
        scaler.unscale_(
            optimizer
        )
    
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRADIENT_CLIP
    )
    
    if use_amp:
    
        scaler.step(
            optimizer
        )
    
        scaler.update()
    
    else:
    
        optimizer.step()
    
    optimizer.zero_grad(
        set_to_none=True
    )


# ============================================================

# 38. TRAIN ONE EPOCH

# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device,
    scaler,
    use_amp
):

    model.train()
    
    optimizer.zero_grad(
        set_to_none=True
    )
    
    total_loss = 0.0
    num_batches = len(dataloader)
    
    progress = tqdm(
        enumerate(
            dataloader,
            start=1
        ),
        total=num_batches,
        desc="Training"
    )
    
    for step, batch in progress:
    
        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )
    
        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )
    
        edge_index = batch["edge_index"].to(
            device,
            non_blocking=True
        )
    
        conversation_ids = batch["conversation_ids"].to(
            device,
            non_blocking=True
        )
    
        graph_stats = batch["graph_stats"].to(
            device,
            non_blocking=True
        )
    
        labels = batch["labels"].to(
            device,
            non_blocking=True
        )
    
        batch_size = labels.size(0)
    
        try:
    
            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
    
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    edge_index=edge_index,
                    conversation_ids=conversation_ids,
                    graph_stats=graph_stats,
                    batch_size=batch_size
                )
    
                raw_loss = criterion(
                    logits,
                    labels
                )
    
                loss = (
                    raw_loss /
                    GRAD_ACCUM_STEPS
                )
    
            if use_amp:
    
                scaler.scale(
                    loss
                ).backward()
    
            else:
    
                loss.backward()
    
            total_loss += raw_loss.item()
    
            should_update = (
                step % GRAD_ACCUM_STEPS == 0
                or
                step == num_batches
            )
    
            if should_update:
    
                optimizer_update(
                    model,
                    optimizer,
                    scaler,
                    use_amp
                )
    
            progress.set_postfix(
                loss=f"{raw_loss.item():.4f}",
                nodes=input_ids.size(0),
                edges=edge_index.size(1)
            )
    
        except torch.OutOfMemoryError:
    
            print("\nCUDA OOM")
            print("Nodes:", input_ids.size(0))
            print("Edges:", edge_index.size(1))
    
            print_cuda_memory()
    
            optimizer.zero_grad(
                set_to_none=True
            )
    
            gc.collect()
    
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
            raise
    
    return (
        total_loss /
        max(num_batches, 1)
    )


# ============================================================

# 39. BUILD MODEL

# ============================================================

def build_model(
    variant,
    model_dir,
    num_classes
):

    if variant == "semantic":
    
        return SemanticOnlyClassifier(
            DATASET,
            model_dir,
            num_classes
        )
    
    if variant == "gat":
    
        return GATOnlyClassifier(
            DATASET,
            model_dir,
            num_classes
        )
    
    if variant == "ssee":
    
        return GATSSEEClassifier(
            DATASET,
            model_dir,
            num_classes
        )
    
    raise ValueError(
        f"Unknown model variant: {variant}"
    )


# ============================================================

# 40. CONTROLLED OPTIMIZER

# ============================================================

def build_optimizer(model):

    encoder_parameters = list(
        model.encoder.parameters()
    )
    
    encoder_ids = {
        id(parameter)
        for parameter in encoder_parameters
    }
    
    new_parameters = [
        parameter
        for parameter in model.parameters()
        if id(parameter)
        not in encoder_ids
    ]
    
    return torch.optim.AdamW(
        [
            {
                "params":
                    encoder_parameters,
                "lr":
                    ENCODER_LR
            },
            {
                "params":
                    new_parameters,
                "lr":
                    NEW_MODULE_LR
            },
        ],
        weight_decay=WEIGHT_DECAY
    )


# ============================================================

# 41. SAVE PREDICTIONS

# ============================================================

def save_predictions(
    sample_ids,
    labels,
    predictions,
    id2label,
    path
):

    rows = []
    
    for sample_id, y_true, y_pred in zip(
        sample_ids,
        labels,
        predictions
    ):
    
        rows.append(
            {
                "sample_id":
                    sample_id,
                "true_id":
                    y_true,
                "true_label":
                    id2label[y_true],
                "pred_id":
                    y_pred,
                "pred_label":
                    id2label[y_pred],
            }
        )
    
    pd.DataFrame(
        rows
    ).to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================

# 42. ONE MODEL × ONE WINDOW

# ============================================================

def run_one_experiment(
    variant,
    window,
    window_splits,
    tokenizer,
    model_dir,
    label2id,
    id2label,
    device,
    use_amp
):

    print("\n" + "=" * 90)
    
    print(
        f"MODEL={variant.upper()} | "
        f"WINDOW={window} MIN | "
        f"SEED={SEED}"
    )
    
    print("=" * 90)
    
    # Reset random state at the start of each controlled run.
    set_seed(SEED)
    
    train_dataset = ConversationGraphDataset(
        window_splits["train"],
        label2id
    )
    
    val_dataset = ConversationGraphDataset(
        window_splits["val"],
        label2id
    )
    
    test_dataset = ConversationGraphDataset(
        window_splits["test"],
        label2id
    )
    
    collator = GraphConversationCollator(
        tokenizer,
        MAX_LENGTH
    )
    
    pin_memory = (
        device.type == "cuda"
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
        pin_memory=pin_memory
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
        pin_memory=pin_memory
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
        pin_memory=pin_memory
    )
    
    model = build_model(
        variant,
        model_dir,
        len(label2id)
    ).to(device)
    
    optimizer = build_optimizer(
        model
    )
    
    criterion = nn.CrossEntropyLoss()
    
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )
    
    run_dir = (
        EARLY_RESULTS_DIR /
        f"{variant}_seed{SEED}" /
        f"{window}min"
    )
    
    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )
    
    best_model_path = (
        run_dir /
        "best_model.pt"
    )
    
    history_path = (
        run_dir /
        "training_history.csv"
    )
    
    best_macro_f1 = -1.0
    patience_counter = 0
    history = []
    
    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):
    
        print(
            f"\nEpoch "
            f"{epoch}/"
            f"{MAX_EPOCHS}"
        )
    
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            scaler,
            use_amp
        )
    
        val_results = evaluate(
            model,
            val_loader,
            device,
            criterion,
            id2label,
            use_amp
        )
    
        print(
            f"Train Loss   : "
            f"{train_loss:.4f}"
        )
    
        print(
            f"Val Loss     : "
            f"{val_results['loss']:.4f}"
        )
    
        print(
            f"Val Accuracy : "
            f"{val_results['accuracy']:.4f}"
        )
    
        print(
            f"Val Macro-F1 : "
            f"{val_results['macro_f1']:.4f}"
        )
    
        history.append(
            {
                "epoch":
                    epoch,
                "train_loss":
                    train_loss,
                "val_loss":
                    val_results["loss"],
                "val_accuracy":
                    val_results["accuracy"],
                "val_macro_f1":
                    val_results["macro_f1"],
            }
        )
    
        pd.DataFrame(
            history
        ).to_csv(
            history_path,
            index=False,
            encoding="utf-8-sig"
        )
    
        if (
            val_results["macro_f1"]
            >
            best_macro_f1
        ):
    
            best_macro_f1 = (
                val_results[
                    "macro_f1"
                ]
            )
    
            patience_counter = 0
    
            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),
                    "epoch":
                        epoch,
                    "best_macro_f1":
                        best_macro_f1,
                    "dataset":
                        DATASET,
                    "window_min":
                        window,
                    "variant":
                        variant,
                    "seed":
                        SEED,
                    "label2id":
                        label2id,
                    "encoder_lr":
                        ENCODER_LR,
                    "new_module_lr":
                        NEW_MODULE_LR,
                    "max_length":
                        MAX_LENGTH,
                    "node_chunk_size":
                        NODE_CHUNK_SIZE,
                    "grad_accum_steps":
                        GRAD_ACCUM_STEPS,
                },
                best_model_path
            )
    
            print(
                "✓ Best checkpoint saved."
            )
    
        else:
    
            patience_counter += 1
    
            print(
                f"No improvement. "
                f"Patience "
                f"{patience_counter}/"
                f"{PATIENCE}"
            )
    
        gc.collect()
    
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
        if patience_counter >= PATIENCE:
    
            print(
                "Early stopping."
            )
    
            break
    
    checkpoint = torch.load(
        best_model_path,
        map_location=device,
        weights_only=False
    )
    
    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )
    
    test_results = evaluate(
        model,
        test_loader,
        device,
        criterion,
        id2label,
        use_amp
    )
    
    print("\nBEST / TEST")
    
    print(
        f"Best Epoch    : "
        f"{checkpoint['epoch']}"
    )
    
    print(
        f"Best Val F1   : "
        f"{checkpoint['best_macro_f1']:.4f}"
    )
    
    print(
        f"Test Accuracy : "
        f"{test_results['accuracy']:.4f}"
    )
    
    print(
        f"Test Macro-F1 : "
        f"{test_results['macro_f1']:.4f}"
    )
    
    report_df = pd.DataFrame(
        test_results[
            "report"
        ]
    ).transpose()
    
    report_df.to_csv(
        run_dir /
        "classification_report.csv",
        encoding="utf-8-sig"
    )
    
    label_ids = list(
        id2label.keys()
    )
    
    cm = confusion_matrix(
        test_results["labels"],
        test_results["predictions"],
        labels=label_ids
    )
    
    cm_df = pd.DataFrame(
        cm,
        index=[
            f"true_{id2label[i]}"
            for i in label_ids
        ],
        columns=[
            f"pred_{id2label[i]}"
            for i in label_ids
        ]
    )
    
    cm_df.to_csv(
        run_dir /
        "confusion_matrix.csv",
        encoding="utf-8-sig"
    )
    
    save_predictions(
        test_results["sample_ids"],
        test_results["labels"],
        test_results["predictions"],
        id2label,
        run_dir /
        "test_predictions.csv"
    )
    
    summary = {
        "dataset":
            DATASET,
        "window_min":
            window,
        "model":
            variant,
        "seed":
            SEED,
        "best_epoch":
            checkpoint["epoch"],
        "best_val_macro_f1":
            checkpoint[
                "best_macro_f1"
            ],
        "test_accuracy":
            test_results[
                "accuracy"
            ],
        "test_macro_f1":
            test_results[
                "macro_f1"
            ],
        "encoder_lr":
            ENCODER_LR,
        "new_module_lr":
            NEW_MODULE_LR,
        "max_length":
            MAX_LENGTH,
        "node_chunk_size":
            NODE_CHUNK_SIZE,
        "effective_batch_size":
            BATCH_SIZE *
            GRAD_ACCUM_STEPS,
    }
    
    for class_id, value in enumerate(
        test_results[
            "class_f1"
        ]
    ):
    
        summary[
            f"test_f1_"
            f"{id2label[class_id]}"
        ] = float(value)
    
    pd.DataFrame(
        [summary]
    ).to_csv(
        run_dir /
        "summary.csv",
        index=False,
        encoding="utf-8-sig"
    )
    
    del model
    del optimizer
    del scaler
    
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return summary



# ============================================================

# 43. RESUME HELPERS

# ============================================================

def load_completed_summary(variant, window):
    summary_path = (
        EARLY_RESULTS_DIR /
        f"{variant}_seed{SEED}" /
        f"{window}min" /
        "summary.csv"
    )

    if not summary_path.exists():
        return None
    
    try:
        df = pd.read_csv(summary_path)
    except Exception as e:
        print(f"Warning: cannot read {summary_path}: {e}")
        return None
    
    if len(df) != 1:
        return None
    
    row = df.iloc[0].to_dict()
    
    required = [
        "dataset",
        "window_min",
        "model",
        "seed",
        "best_epoch",
        "best_val_macro_f1",
        "test_accuracy",
        "test_macro_f1",
    ]
    
    if any(key not in row for key in required):
        return None
    
    try:
        if int(row["window_min"]) != int(window):
            return None
        if int(row["seed"]) != int(SEED):
            return None
        if str(row["model"]) != str(variant):
            return None
    except Exception:
        return None
    
    return row


def normalize_summary_row(row):
    result = {}
    for key, value in row.items():
        if pd.isna(value):
            result[key] = None
        else:
            result[key] = value
    return result


def rebuild_aggregate_files(all_results):
    if not all_results:
        return

    results_df = pd.DataFrame(all_results)
    
    model_order = {
        "semantic": 0,
        "gat": 1,
        "ssee": 2,
    }
    
    results_df["_model_order"] = (
        results_df["model"]
        .map(model_order)
    )
    
    results_df = (
        results_df
        .sort_values(
            by=["window_min", "_model_order"]
        )
        .drop(columns=["_model_order"])
    )
    
    results_df.to_csv(
        EARLY_RESULTS_DIR / "early_window_results.csv",
        index=False,
        encoding="utf-8-sig"
    )
    
    pivot_f1 = results_df.pivot(
        index="model",
        columns="window_min",
        values="test_macro_f1"
    )
    
    pivot_f1.to_csv(
        EARLY_RESULTS_DIR / "macro_f1_by_window.csv",
        encoding="utf-8-sig"
    )
    
    pivot_acc = results_df.pivot(
        index="model",
        columns="window_min",
        values="test_accuracy"
    )
    
    pivot_acc.to_csv(
        EARLY_RESULTS_DIR / "accuracy_by_window.csv",
        encoding="utf-8-sig"
    )


# ============================================================

# 44. MAIN - RESUME MODE

# ============================================================

def main():

    global DATASET
    DATASET = "DRWeibo"
    
    set_seed(SEED)
    
    device = get_device()
    use_amp = device.type == "cuda"
    
    print("\n" + "=" * 90)
    print("DRWEIBO EARLY-WINDOW EXPERIMENT - RESUME MODE")
    print("=" * 90)
    
    print("Windows:", WINDOWS)
    print("Models :", MODEL_VARIANTS)
    print("Seed   :", SEED)
    print(
        "Valid existing summary.csv files will be skipped."
    )
    
    split_ids = load_fixed_split_ids()
    
    print("\nLocked split sizes:")
    for split_name in ["train", "val", "test"]:
        print(
            f"  {split_name:5s}: "
            f"{len(split_ids[split_name])}"
        )
    
    model_dir = DRWEIBO_MODEL_DIR
    
    if not model_dir.exists():
        raise FileNotFoundError(model_dir)
    
    tokenizer = load_tokenizer(
        DATASET,
        model_dir
    )
    
    label2id, id2label = get_label_mapping(
        DATASET
    )
    
    EARLY_RESULTS_DIR.mkdir(
        parents=True,
        exist_ok=True
    )
    
    all_results = []
    
    print("\nScanning completed runs...")
    
    for window in WINDOWS:
        for variant in MODEL_VARIANTS:
    
            completed = load_completed_summary(
                variant,
                window
            )
    
            if completed is not None:
                completed = normalize_summary_row(
                    completed
                )
                all_results.append(completed)
    
                print(
                    f"  ✓ SKIP {variant:8s} "
                    f"{window:3d}min | "
                    f"Macro-F1="
                    f"{float(completed['test_macro_f1']):.4f}"
                )
            else:
                print(
                    f"  · TODO {variant:8s} "
                    f"{window:3d}min"
                )
    
    rebuild_aggregate_files(
        all_results
    )
    
    for window in WINDOWS:
    
        pending = [
            variant
            for variant in MODEL_VARIANTS
            if load_completed_summary(
                variant,
                window
            ) is None
        ]
    
        if not pending:
            print(
                f"\n{window}min fully complete -> skip window."
            )
            continue
    
        window_path = get_early_window_path(
            window
        )
    
        print("\n" + "-" * 90)
        print(
            f"Preparing {window}-min data:"
        )
        print(window_path)
    
        window_splits = build_window_splits(
            window_path,
            split_ids
        )
    
        print(
            "Window split sizes:",
            {
                k: len(v)
                for k, v in window_splits.items()
            }
        )
    
        for variant in MODEL_VARIANTS:
    
            if load_completed_summary(
                variant,
                window
            ) is not None:
                print(
                    f"SKIP: {variant} {window}min already done."
                )
                continue
    
            print(
                f"\nSTART: {variant} {window}min"
            )
    
            result = run_one_experiment(
                variant=variant,
                window=window,
                window_splits=window_splits,
                tokenizer=tokenizer,
                model_dir=model_dir,
                label2id=label2id,
                id2label=id2label,
                device=device,
                use_amp=use_amp
            )
    
            all_results = [
                row
                for row in all_results
                if not (
                    str(row.get("model")) == str(variant)
                    and
                    int(row.get("window_min")) == int(window)
                )
            ]
    
            all_results.append(result)
    
            rebuild_aggregate_files(
                all_results
            )
    
            print(
                f"✓ Finished {variant} {window}min"
            )
    
    final_results = []
    
    for window in WINDOWS:
        for variant in MODEL_VARIANTS:
    
            completed = load_completed_summary(
                variant,
                window
            )
    
            if completed is not None:
                final_results.append(
                    normalize_summary_row(completed)
                )
    
    rebuild_aggregate_files(
        final_results
    )
    
    results_df = pd.DataFrame(
        final_results
    )
    
    print("\n" + "=" * 90)
    print("RESUME RUN FINISHED")
    print("=" * 90)
    
    if len(results_df) > 0:
    
        pivot_f1 = results_df.pivot(
            index="model",
            columns="window_min",
            values="test_macro_f1"
        )
    
        pivot_acc = results_df.pivot(
            index="model",
            columns="window_min",
            values="test_accuracy"
        )
    
        print("\nMacro-F1 by completed window:")
        print(pivot_f1.round(4))
    
        print("\nAccuracy by completed window:")
        print(pivot_acc.round(4))
    
    expected_runs = (
        len(WINDOWS) *
        len(MODEL_VARIANTS)
    )
    
    print(
        f"\nCompleted runs: "
        f"{len(final_results)}/"
        f"{expected_runs}"
    )
    
    print("\nAggregated files:")
    print(
        EARLY_RESULTS_DIR /
        "early_window_results.csv"
    )
    print(
        EARLY_RESULTS_DIR /
        "macro_f1_by_window.csv"
    )
    print(
        EARLY_RESULTS_DIR /
        "accuracy_by_window.csv"
    )


if __name__ == "__main__":
    main()